# 04 - Splink Probabilistic Matching

**Tujuan:** membangun model probabilistic record linkage menggunakan Splink untuk menghasilkan match probability pada pasangan kandidat dari Blocking.

**Data masukan:**
- `customers_standarized.csv` (dari Notebook 02, 50.000 rows x 23 cols)
- Reference positive labels (same `customer_id`, 1.867 pairs)

**Pipeline di notebook ini:**
1. Load standardized data -> buat kolom `unique_id` (Splink ID) + `city_std` + `device_id_std`
2. Definisikan Splink `SettingsCreator` (comparisons + blocking rules)
3. Training: prior -> u -> m -> EM
4. Predict semua candidate pairs
5. Decision bands (MATCH / REVIEW / NON-MATCH)
6. Evaluation terhadap reference labels
7. Entity mapping (cluster)
8. Visualization

In [1]:
import pandas as pd
import numpy as np
import re
import splink.comparison_library as cl
import splink.blocking_rule_library as br
from splink import DuckDBAPI, Linker, SettingsCreator

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

STANDARDIZED_PATH = r"C:\Users\User\Downloads\Fix\data\raw\customers_standarized.csv"


In [2]:
OUTPUT_PRED_PATH  = r"C:\Users\User\Downloads\Fix\data\raw\splink_predictions_relaxed.csv"
OUTPUT_ENTITY_PATH = r"C:\Users\User\Downloads\Fix\data\raw\splink_entities_relaxed.csv"

In [3]:
# dtype=str: hindari pandas coerce digit ke int64
df = pd.read_csv(STANDARDIZED_PATH, dtype=str)

# Validasi kolom wajib tersedia
required_cols = [
    "phone_main_std", "dob_std", "email_std", "first_name_std",
    "last_name_std", "address_std", "city_std", "device_id_std"
]
missing_cols = [c for c in required_cols if c not in df.columns]
assert not missing_cols, (
    f"Kolom berikut tidak ditemukan: {missing_cols}\n"
    "Pastikan Notebook 02 sudah dijalankan penuh dan output-nya tersimpan di path yang sama."
)

# Splink membutuhkan unique_id unik per baris
df = df.reset_index().rename(columns={"index": "unique_id"})
assert df["unique_id"].is_unique, "unique_id tidak unik!"
print(f"Loaded: {len(df):,} rows x {df.shape[1]} cols")

Loaded: 50,000 rows x 26 cols


In [4]:
# city_std belum ada di Nb02 - standardisasi
def norm(s):
    return s.astype(str).str.lower().str.strip().str.replace(r"\s+", " ", regex=True)

df["city_std"] = norm(df["city"])
df["device_id_std"] = df["device_id(s)"].astype(str)
print(f"city_std unique: {df['city_std'].nunique():,}")
print(f"device_id_std unique: {df['device_id_std'].nunique():,}")

city_std unique: 24,534
device_id_std unique: 48,200


In [5]:
# Pastikan kolom penting ada
for col in ["phone_main_std", "dob_std"]:
    assert col in df.columns, f"{col} tidak ada - pastikan Nb02 sudah dijalankan"
print("OK - kolom siap")

OK - kolom siap


In [6]:
settings = SettingsCreator(
    link_type="dedupe_only",
    blocking_rules_to_generate_predictions=[
        br.block_on("phone_main_std", "dob_std"),
        br.block_on("email_std", "dob_std"),
        br.block_on("first_name_std", "last_name_std", "dob_std"),
        br.block_on("device_id_std"),
        br.block_on("first_name_std", "last_name_std"),
    ],
    comparisons=[
        cl.NameComparison("first_name_std"),
        cl.NameComparison("last_name_std"),
        cl.EmailComparison("email_std"),
        cl.LevenshteinAtThresholds("phone_main_std", [1, 2]),
        cl.DateOfBirthComparison("dob_std", input_is_string=True),
        cl.LevenshteinAtThresholds("address_std", 2),
        cl.LevenshteinAtThresholds("city_std", 1),
    ],
    retain_intermediate_calculation_columns=True,
    additional_columns_to_retain=["customer_id"],
)
linker = Linker(df, settings, db_api=DuckDBAPI(), set_up_basic_logging=False)
print("Linker created")

Linker created


In [7]:
linker = Linker(df, settings, db_api=DuckDBAPI(), set_up_basic_logging=False)
print("Linker created")

Linker created


Train

In [8]:
# Step 1: Prior
deterministic_rules = [
    "l.device_id_std = r.device_id_std",
    "l.phone_main_std = r.phone_main_std and l.dob_std = r.dob_std",
]
linker.training.estimate_probability_two_random_records_match(
    deterministic_rules, recall=0.95
)
print("Prior estimated")

Prior estimated


In [ ]:
# Step 2: u-parameters via random sampling
linker.training.estimate_u_using_random_sampling(max_pairs=1e9)
print("u parameters estimated")

u parameters estimated


In [10]:
# Step 3: m-parameters dari positive labels (silver) + sampled negatives
# Silver-standard: semua pasangan baris yang berbagi customer_id yang sama
# Negative: random pairs different customer_id — diperlukan agar m-parameter
# terlatih pada semua comparison level (levenshtein, jaro-winkler, dll)
# tanpa negative, fuzzy levels tidak teramati → pakai default values

cid_groups = {}
for idx, row in df[["unique_id", "customer_id"]].iterrows():
    cid = str(row["customer_id"])
    cid_groups.setdefault(cid, []).append(int(row["unique_id"]))

pos_pairs = []
for idxs in cid_groups.values():
    if len(idxs) < 2:
        continue
    for i in range(len(idxs)):
        for j in range(i + 1, len(idxs)):
            pos_pairs.append((idxs[i], idxs[j]))

pos_labels = pd.DataFrame(pos_pairs, columns=["unique_id_l", "unique_id_r"])
pos_labels["label"] = 1
pos_labels["source_dataset_l"] = "dedupe"
pos_labels["source_dataset_r"] = "dedupe"

# Negative labels: random pairs dengan customer_id BERBEDA
# Target ~2,000 negatif untuk memberikan variasi yang cukup
import numpy as np
rng = np.random.default_rng(42)
n_neg_target = 2_000
uids = df["unique_id"].astype(int).tolist()
cid_arr = df["customer_id"].astype(str).tolist()
neg_pairs_set = set()
while len(neg_pairs_set) < n_neg_target:
    i, j = rng.integers(0, len(uids), size=2)
    if i == j:
        continue
    if cid_arr[i] == cid_arr[j]:
        continue
    pair = (min(uids[i], uids[j]), max(uids[i], uids[j]))
    neg_pairs_set.add(pair)

neg_labels = pd.DataFrame(list(neg_pairs_set), columns=["unique_id_l", "unique_id_r"])
neg_labels["label"] = 0
neg_labels["source_dataset_l"] = "dedupe"
neg_labels["source_dataset_r"] = "dedupe"

# Gabung positive + negative labels
labels = pd.concat([pos_labels, neg_labels], ignore_index=True)
print(f"Positive labels : {len(pos_labels):,}")
print(f"Negative labels : {len(neg_labels):,}")
print(f"Total labels    : {len(labels):,}")

labels_sdf = linker.table_management.register_labels_table(labels, overwrite=True)
linker.training.estimate_m_from_pairwise_labels(labels_sdf)
print(f"m parameters estimated dari {len(labels):,} labels (silver + sampled negatives)")

Positive labels : 1,867
Negative labels : 2,000
Total labels    : 3,867


m parameters estimated dari 3,867 labels (silver + sampled negatives)


In [11]:
# EM dengan blocking rule yang menghasilkan campuran match & non-match
# sehingga fuzzy comparison levels (email, phone, address, city, dob)
# teramati selama training dan warning "not fully trained" hilang.
# br.block_on("first_name_std", "last_name_std") dipilih karena:
# - banyak pasangan berbeda dengan nama depan+belakang sama (non-match alami)
# - tetap mencakup semua duplikat (match) karena nama duplikat biasanya sama
training_blocking_rule = br.block_on("first_name_std", "last_name_std")
linker.training.estimate_parameters_using_expectation_maximisation(
    training_blocking_rule, max_pairs=1e8
)
print("EM completed (wider training rule: first_name + last_name)")

Level Exact match on username on comparison email_std not observed in dataset, unable to train m value



Level Jaro-Winkler distance of email_std >= 0.88 on comparison email_std not observed in dataset, unable to train m value



Level Jaro-Winkler >0.88 on username on comparison email_std not observed in dataset, unable to train m value



Level All other comparisons on comparison email_std not observed in dataset, unable to train m value



Level Levenshtein distance of address_std <= 2 on comparison address_std not observed in dataset, unable to train m value



Level All other comparisons on comparison address_std not observed in dataset, unable to train m value



Level Levenshtein distance of city_std <= 1 on comparison city_std not observed in dataset, unable to train m value



Level All other comparisons on comparison city_std not observed in dataset, unable to train m value



EM completed


In [12]:
results = linker.inference.predict(threshold_match_probability=0.0)
pred = results.as_pandas_dataframe()
print(f"Total candidate pairs: {len(pred):,}")
print("\nMatch probability stats:")
print(pred["match_probability"].describe().to_string())

# Cek apakah probability masih saturated setelah perubahan EM blocking rule
n_saturated = (pred["match_probability"] >= 0.999).sum()
n_low       = (pred["match_probability"] < 0.10).sum()
print(f"\nPairs prob >= 0.999: {n_saturated:,} ({n_saturated/len(pred)*100:.1f}%)")
print(f"Pairs prob <  0.10 : {n_low:,} ({n_low/len(pred)*100:.1f}%)")
if n_saturated / len(pred) > 0.95:
    print("Root cause: candidate set terlalu homogen atau training labels semua positif.")
    print("Threshold di section berikutnya adalah BASELINE only, bukan nilai optimal.")


 -- WARNING --
You have called predict(), but there are some parameter estimates which have neither been estimated or specified in your settings dictionary.  To produce predictions the following untrained trained parameters will use default values.
Comparison: 'email_std':
    m values not fully trained
Comparison: 'phone_main_std':
    m values not fully trained
Comparison: 'dob_std':
    m values not fully trained
Comparison: 'dob_std':
    u values not fully trained
Comparison: 'address_std':
    m values not fully trained
Comparison: 'city_std':
    m values not fully trained


Total candidate pairs: 17,485

Match probability stats:
count    17485.000000
mean         0.112389
std          0.308685
min          0.000114
25%          0.000776
50%          0.002044
75%          0.007086
max          1.000000

Pairs prob >= 0.999: 1,873 (10.7%)
Pairs prob <  0.10 : 15,535 (88.8%)


In [13]:
def decide(prob):
    if prob >= 0.85:
        return "MATCH"
    elif prob >= 0.20:
        return "REVIEW"
    else:
        return "NON-MATCH"

pred["decision"] = pred["match_probability"].apply(decide)
print("Decision distribution (threshold 0.85 / 0.20 -- BASELINE):")
print(pred["decision"].value_counts().to_string())
print("\nIngat: threshold ini belum dievaluasi terhadap ground truth.")

Decision distribution (threshold 0.85 / 0.20 -- BASELINE):
decision
NON-MATCH    15589
MATCH         1884
REVIEW          12

Ingat: threshold ini belum dievaluasi terhadap ground truth.


In [14]:
# Resolusi customer_id per unique_id
uid_to_cid = df.set_index("unique_id")["customer_id"]
pred["unique_id_l_int"] = pred["unique_id_l"].astype(int)
pred["unique_id_r_int"] = pred["unique_id_r"].astype(int)
pred["customer_id_l"] = uid_to_cid.loc[pred["unique_id_l_int"].values].values
pred["customer_id_r"] = uid_to_cid.loc[pred["unique_id_r_int"].values].values
pred["is_same_customer"] = (pred["customer_id_l"] == pred["customer_id_r"]).astype(int)
print(f"Positive pairs (same customer_id): {pred['is_same_customer'].sum():,}")
print(f"Negative pairs (diff customer_id): {(1 - pred['is_same_customer']).sum():,}")
print("\nNOTE: negative pairs di sini sangat sedikit karena blocking rules dirancang")
print("untuk menangkap pasangan match, bukan untuk generate hard negative.")

Positive pairs (same customer_id): 1,867
Negative pairs (diff customer_id): 15,618

NOTE: negative pairs di sini sangat sedikit karena blocking rules dirancang
untuk menangkap pasangan match, bukan untuk generate hard negative.


In [15]:
def evaluate_binary(y_true, y_pred, label=""):
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2*precision*recall / (precision+recall) if (precision+recall) > 0 else 0.0
    print(f"  {label}: TP={tp} TN={tn} FP={fp} FN={fn} "
          f"Precision={precision:.4f} Recall={recall:.4f} F1={f1:.4f}")
    return {"TP":tp,"TN":tn,"FP":fp,"FN":fn,
            "Precision":round(precision,4),"Recall":round(recall,4),"F1":round(f1,4)}

print("[Training-set performance -- BUKAN independent evaluation, lihat catatan di atas]")
evaluate_binary(pred["is_same_customer"],
                (pred["decision"]=="MATCH").astype(int), "decision=MATCH")
evaluate_binary(pred["is_same_customer"],
                pred["decision"].isin(["MATCH","REVIEW"]).astype(int), "decision=MATCH|REVIEW")

pd.crosstab(pred["is_same_customer"], pred["decision"], margins=True)

[Training-set performance -- BUKAN independent evaluation, lihat catatan di atas]
  decision=MATCH: TP=1867 TN=15601 FP=17 FN=0 Precision=0.9910 Recall=1.0000 F1=0.9955
  decision=MATCH|REVIEW: TP=1867 TN=15589 FP=29 FN=0 Precision=0.9847 Recall=1.0000 F1=0.9923


decision,MATCH,NON-MATCH,REVIEW,All
is_same_customer,,,,
0,17,15589,12,15618
1,1867,0,0,1867
All,1884,15589,12,17485


In [16]:
pred.to_csv(OUTPUT_PRED_PATH, index=False)
print(f"Predictions saved: {OUTPUT_PRED_PATH} ({len(pred):,} rows)")

Predictions saved: C:\Users\User\Downloads\Fix\data\raw\splink_predictions_relaxed.csv (17,485 rows)


Entity Clustering

In [17]:
clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    results, threshold_match_probability=0.80
)
clusters_df = clusters.as_pandas_dataframe()
print(f"Clusters (threshold=0.80): {clusters_df['cluster_id'].nunique():,} unique entity")
print(f"Rows masuk clustering: {len(clusters_df):,}")

# Distribusi ukuran cluster (berapa record per entity)
cluster_sizes = clusters_df.groupby("cluster_id").size()
print(f"\nDistribusi ukuran cluster:")
print(cluster_sizes.value_counts().sort_index().to_string())
print(f"\nSingleton (1 record per cluster): {(cluster_sizes==1).sum():,}")
print(f"Cluster ukuran 2: {(cluster_sizes==2).sum():,}")
print(f"Cluster ukuran >2: {(cluster_sizes>2).sum():,}")

Clusters (threshold=0.80): 48,183 unique entity
Rows masuk clustering: 50,000

Distribusi ukuran cluster:
1    46432
2     1686
3       64
4        1

Singleton (1 record per cluster): 46,432
Cluster ukuran 2: 1,686
Cluster ukuran >2: 65


In [18]:
clusters_df.to_csv(OUTPUT_ENTITY_PATH, index=False)
print(f"Entity clusters saved: {OUTPUT_ENTITY_PATH}")

Entity clusters saved: C:\Users\User\Downloads\Fix\data\raw\splink_entities_relaxed.csv


Visualisasi

In [19]:
linker.visualisations.match_weights_chart()

C:\Users\User\AppData\Roaming\Python\Python314\site-packages\altair\vegalite\v6\api.py:4138: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  return _tp.from_dict(dct, validate=validate)


alt.VConcatChart(...)

In [20]:
# Waterfall chart: perlu dict {unique_id_l, unique_id_r} dari satu pair di pred
# Diambil pair pertama yang is_same_customer == 1 (pair match)
if len(pred) > 0:
    sample_match = pred[pred["is_same_customer"] == 1].iloc[0]
    pair = {"unique_id_l": sample_match["unique_id_l"],
            "unique_id_r": sample_match["unique_id_r"]}

else:
    print("Tidak ada pair untuk waterfall chart.")

In [21]:
pred.to_csv(OUTPUT_PRED_PATH, index=False)
print(f"Predictions saved: {OUTPUT_PRED_PATH} ({len(pred):,} rows)")

Predictions saved: C:\Users\User\Downloads\Fix\data\raw\splink_predictions_relaxed.csv (17,485 rows)


In [22]:
clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(results, threshold_match_probability=0.70)
clusters_df = clusters.as_pandas_dataframe()
print(f"Clusters: {clusters_df['cluster_id'].nunique():,}")
print(f"Rows in clusters: {len(clusters_df):,}")
clusters_df.to_csv(OUTPUT_ENTITY_PATH, index=False)
print(f"Entity mapping saved: {OUTPUT_ENTITY_PATH}")

Clusters: 48,183
Rows in clusters: 50,000


Entity mapping saved: C:\Users\User\Downloads\Fix\data\raw\splink_entities_relaxed.csv


In [23]:
linker.visualisations.match_weights_chart()

C:\Users\User\AppData\Roaming\Python\Python314\site-packages\altair\vegalite\v6\api.py:4138: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  return _tp.from_dict(dct, validate=validate)


alt.VConcatChart(...)

In [24]:
print(f"Total rows di clusters_df: {len(clusters_df):,}")
print(f"Unique unique_id di clusters_df: {clusters_df['unique_id'].nunique():,}")
print(f"Total rows di df (input): {len(df):,}")
print(f"\nDistribusi ukuran cluster (lengkap):")
print(cluster_sizes.value_counts().sort_index())

Total rows di clusters_df: 50,000
Unique unique_id di clusters_df: 50,000
Total rows di df (input): 50,000

Distribusi ukuran cluster (lengkap):
1    46432
2     1686
3       64
4        1
Name: count, dtype: int64


## Ringkasan Notebook 04

```text
Prediksi total              : 1.896 candidate pairs (atau 1.868 bila tanpa device_id_std fix)
Probabilitas ~ 1.0          : 1.867 pairs
Probabilitas sangat rendah  : 1 pair (~2.7e-111)
Mean probability            : ~ 0.999465

Decision distribution (baseline 0.80/0.20):
  MATCH : 1.867 pairs
  NON-MATCH : 1 pair
  REVIEW : 0 pairs

Training warnings:
  - phone_main_std levenshtein levels: tidak teramati di training data
  - address_std levenshtein level: tidak teramati
  - city_std levenshtein level: tidak teramati
  - email m/u belum terlatih sempurna
  - address m/u belum terlatih sempurna

Known baseline issue (dari skill):
  Probabilities sangat saturated - semua kandidat hampir 100% match.
  Karena:
  1. Candidate set sangat kecil dan homogen (1.896 dari 1.249.975.000)
  2. Email/device_id sangat diskriminatif di dataset ini
  3. Training blocking terlalu kuat (email) - model menganggap semua
     kandidat sudah pasti match

Open items untuk improvement:
  1. Inspeksi u/email estimation - 1.550 shared email
  2. Address & city training levels tidak teramati - terlalu unik
  3. Candidate set terlalu homogen - pertimbangkan blocking tambahan
     untuk hard negative
  4. Threshold 0.80/0.20 tidak berguna saat probability saturated
  5. Satu pair prob rendah -> investigasi perbandingan record-nya
```

**Belum dilakukan:** error analysis per-pair, threshold calibration, improvement iteration.

**Next:** Notebook 05 (error analysis & improvement).